# NPCore — Advanced Demo

Este notebook presenta una demostración avanzada de **npcore**, una librería de Python diseñada para modelar comportamiento de NPCs (Non-Playable Characters) mediante decisiones probabilísticas basadas en estado y contexto.
El demo2 está orientado en NPCore como motor de simulación de agentes, no solo ejemplos 

## ¿Qué demuestra este notebook?

A lo largo de este demo veremos cómo:

- Definir sistemas de decisión reutilizables
- Modelar múltiples tipos de NPCs con distintas lógicas
- Ejecutar simulaciones en múltiples pasos
- Comparar comportamientos emergentes

## Idea central

En lugar de decisiones deterministas (if/else rígido), los NPCs:
toman decisiones probabilísticas  
reaccionan al contexto  
generan comportamientos variados

## Casos incluidos

1. Sistema de combate
2. Guardia patrullando
3. Aldeano en peligro
4. Simulación multi-agente


## 1. Configuración del entorno

In [1]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

print("Entorno configurado correctamente.")

Entorno configurado correctamente.


## 2. Importar componentes principales

In [2]:
from npcore.brain import Brain
from npcore.npc import NPC
from npcore.environment import Environment

print("Componentes cargados.")

Componentes cargados.


## 3. Concepto clave: Brain + Contexto

En `npcore`, cada NPC toma decisiones usando:

- **Estado** → define qué tipo de comportamiento usar
- **Contexto** → información del entorno (vida, peligro, etc.)
- **Brain** → conjunto de reglas probabilísticas

Cada regla devuelve un diccionario:

```python
{"acción": probabilidad}

In [3]:
brain_basic = Brain()

def simple_rule(ctx):
    return {"idle": 0.5, "walk": 0.5}

brain_basic.add_rule("default", simple_rule)

npc = NPC("TestNPC", brain_basic)
npc.set_state("default")

print("Acción generada:", npc.act())

Acción generada: idle


### Interpretación

Incluso con una regla simple, el comportamiento no es fijo.
Cada ejecución puede producir resultados distintos.

## 4. Ejemplo básico: decisiones simples

En este primer ejemplo creamos un NPC con una regla simple para entender cómo funciona el sistema de decisión probabilística.

## 5. Sistema de combate dinámico

Ahora usamos contexto (`hp`) para modificar el comportamiento.

In [4]:
brain_combat = Brain()

def combat_rule(ctx):
    hp = ctx.get("hp", 100)

    if hp < 30:
        return {"flee": 0.8, "defend": 0.2}
    elif hp < 60:
        return {"attack": 0.4, "defend": 0.6}
    else:
        return {"attack": 0.8, "defend": 0.2}

brain_combat.add_rule("combat", combat_rule)

In [5]:
goblin = NPC("Goblin", brain_combat)
knight = NPC("Knight", brain_combat)

goblin.set_state("combat")
knight.set_state("combat")

goblin.update_context(hp=20)
knight.update_context(hp=90)

print("Goblin:", goblin.act())
print("Knight:", knight.act())

Goblin: flee
Knight: attack


### Interpretación

- El Goblin tiende a huir
- El Knight tiende a atacar

Mismo sistema, distinto comportamiento.

## 6. Simulación en múltiples pasos

Ahora no queremos observar una sola decisión aislada, sino el comportamiento de varios NPCs a lo largo del tiempo.

Para ello, agregamos los personajes a un `Environment` y ejecutamos varios pasos de simulación.

In [6]:
env_combat = Environment()
env_combat.add_npc(goblin)
env_combat.add_npc(knight)

for step in range(5):
    results = env_combat.step()
    print(f"Paso {step + 1}: {results}")

Paso 1: [('Goblin', 'flee'), ('Knight', 'defend')]
Paso 2: [('Goblin', 'defend'), ('Knight', 'attack')]
Paso 3: [('Goblin', 'flee'), ('Knight', 'attack')]
Paso 4: [('Goblin', 'flee'), ('Knight', 'attack')]
Paso 5: [('Goblin', 'flee'), ('Knight', 'attack')]


### Interpretación

En esta simulación podemos observar que:

- El `Goblin`, al tener poca vida, tiende a elegir acciones defensivas o de escape
- El `Knight`, al tener mucha vida, tiende a elegir acciones ofensivas

Como la decisión es probabilística, el resultado exacto puede variar entre ejecuciones, pero la tendencia general se mantiene.

## 7. Segundo escenario: un guardia patrullando

El siguiente ejemplo muestra que `npcore` no está limitado al combate.

Ahora modelaremos un guardia cuyo comportamiento depende de si detecta o no una amenaza cercana.

In [7]:
brain_guard = Brain()

def guard_rule(ctx):
    enemy_near = ctx.get("enemy_near", False)

    if enemy_near:
        return {"attack": 0.6, "alert": 0.4}
    return {"patrol": 0.7, "rest": 0.3}

brain_guard.add_rule("guard", guard_rule)

guard = NPC("Guard", brain_guard)
guard.set_state("guard")
guard.update_context(enemy_near=False)

print("Guardia creado correctamente.")
print("Contexto inicial:", guard.context)
print("Acción inicial:", guard.act())

Guardia creado correctamente.
Contexto inicial: {'enemy_near': False}
Acción inicial: patrol


### Cambio de contexto

Una ventaja del sistema es que el mismo NPC puede comportarse distinto si cambia el contexto.

A continuación actualizamos la percepción del guardia para simular que detecta un enemigo.

In [8]:
guard.update_context(enemy_near=True)

print("Nuevo contexto:", guard.context)
print("Nueva acción:", guard.act())

Nuevo contexto: {'enemy_near': True}
Nueva acción: attack


### Interpretación

Cuando no hay amenaza, el guardia tiende a patrullar o descansar.

Cuando aparece un enemigo, cambia su comportamiento hacia acciones más reactivas como atacar o alertar.

## 8. Tercer escenario: aldeano en situación de riesgo

En este ejemplo construiremos un NPC con un rol completamente distinto: un aldeano.

La idea es mostrar que la librería puede modelar agentes no combatientes, con lógicas propias.

In [9]:
brain_villager = Brain()

def villager_rule(ctx):
    danger = ctx.get("danger", False)

    if danger:
        return {"hide": 0.8, "run": 0.2}
    return {"work": 0.6, "walk": 0.4}

brain_villager.add_rule("villager", villager_rule)

villager = NPC("Villager", brain_villager)
villager.set_state("villager")
villager.update_context(danger=False)

print("Aldeano creado correctamente.")
print("Acción sin peligro:", villager.act())

Aldeano creado correctamente.
Acción sin peligro: work


### Cambio de contexto

Ahora simulamos que el aldeano percibe peligro.

In [10]:
villager.update_context(danger=True)

print("Contexto actualizado:", villager.context)
print("Acción con peligro:", villager.act())

Contexto actualizado: {'danger': True}
Acción con peligro: run


### Interpretación

Este ejemplo muestra que `npcore` puede reutilizarse para comportamientos civiles, sociales o de supervivencia, no solo para combate.

## 9. Escenario multi-agente

Finalmente, vamos a reunir varios NPCs en un mismo entorno para observar cómo se comportan juntos en una simulación simple.

In [11]:
multi_env = Environment()
multi_env.add_npc(goblin)
multi_env.add_npc(knight)
multi_env.add_npc(guard)
multi_env.add_npc(villager)

for step in range(5):
    results = multi_env.step()
    print(f"Paso {step + 1}: {results}")

Paso 1: [('Goblin', 'flee'), ('Knight', 'defend'), ('Guard', 'attack'), ('Villager', 'hide')]
Paso 2: [('Goblin', 'flee'), ('Knight', 'attack'), ('Guard', 'attack'), ('Villager', 'hide')]
Paso 3: [('Goblin', 'flee'), ('Knight', 'defend'), ('Guard', 'attack'), ('Villager', 'hide')]
Paso 4: [('Goblin', 'defend'), ('Knight', 'attack'), ('Guard', 'attack'), ('Villager', 'run')]
Paso 5: [('Goblin', 'flee'), ('Knight', 'defend'), ('Guard', 'attack'), ('Villager', 'hide')]


### Interpretación

Aquí observamos varios tipos de agentes coexistiendo en un mismo entorno:

- Un NPC de combate con poca vida
- Un NPC de combate con mucha vida
- Un guardia reactivo al entorno
- Un aldeano con lógica de supervivencia

Esto ayuda a demostrar que la librería tiene un diseño reusable y extensible.

## 10. Comparación de comportamientos

Una de las ideas más importantes de `npcore` es que el comportamiento emerge de la combinación de:

- estado
- contexto
- reglas probabilísticas

No hay una única salida fija.  
En cambio, cada ejecución conserva una tendencia general, pero con variación natural en las acciones.

In [12]:
npcs = [goblin, knight, guard, villager]

for npc in npcs:
    print(f"\nNPC: {npc.name}")
    print(f"Estado: {npc.state}")
    print(f"Contexto: {npc.context}")
    print("Acciones observadas:")
    for _ in range(5):
        print("-", npc.act())
        


NPC: Goblin
Estado: combat
Contexto: {'hp': 20}
Acciones observadas:
- flee
- flee
- flee
- flee
- flee

NPC: Knight
Estado: combat
Contexto: {'hp': 90}
Acciones observadas:
- attack
- attack
- attack
- defend
- attack

NPC: Guard
Estado: guard
Contexto: {'enemy_near': True}
Acciones observadas:
- attack
- attack
- alert
- alert
- attack

NPC: Villager
Estado: villager
Contexto: {'danger': True}
Acciones observadas:
- hide
- run
- hide
- run
- hide


## 11. Conclusión

En este notebook mostramos cómo usar `npcore` para:

- definir sistemas de decisión probabilística
- modelar múltiples tipos de NPCs
- cambiar el comportamiento en función del contexto
- ejecutar simulaciones por pasos en un entorno

En conjunto, esto convierte a `npcore` en una base útil para:

- prototipos de videojuegos
- simulaciones educativas
- experimentos con agentes autónomos simples
- sistemas ligeros de comportamiento artificial

## 12. Posibles extensiones

Algunas mejoras futuras para esta librería podrían ser:

- memoria interna del NPC
- interacción directa entre agentes
- mapas o grids de posición
- eventos aleatorios del entorno
- estrategias adaptativas más complejas

Estas extensiones permitirían evolucionar `npcore` hacia simulaciones más ricas y realistas.